In [3]:
import chromadb
import os
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

# experiments 폴더 안에서 실행 중이므로 상위 폴더의 .env를 로드
load_dotenv("../.env")

# 1. ChromaDB 클라이언트 연결
# Jupyter 환경이 experiments 폴더 내라면 데이터베이스 경로는 상위 폴더인 ../naver_pay_db 입니다
client = chromadb.PersistentClient(path="../naver_pay_db")

# 2. 임베딩 함수 설정 (ingest.py와 동일하게 설정)
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name='text-embedding-3-small'
)

# 3. 컬렉션 가져오기 (ingest.py에서 컬렉션 이름을 refund_policy로 생성했습니다)
collection = client.get_collection(name="refund_policy", embedding_function=openai_ef)

# --- 청크(문서) 수 확인 방법 ---

# 방법 A: 질문하신 get() 메서드를 사용하여 확인하는 코드
all_chunks = collection.get()
chunk_count_by_get = len(all_chunks['ids'])
print(f"collection.get()으로 확인한 현재 청크 수: {chunk_count_by_get}개")

# 방법 B: count() 메서드를 사용하는 코드 (권장됨)
chunk_count_by_count = collection.count()
print(f"collection.count()로 확인한 현재 청크 수: {chunk_count_by_count}개")

collection.get()으로 확인한 현재 청크 수: 108개
collection.count()로 확인한 현재 청크 수: 108개


In [ ]:
import sys
import os

# experiments 폴더에서 실행하기 때문에 최상위 폴더(프로젝트 루트)를 모듈 경로에 추가
sys.path.append(os.path.abspath('..'))

from core.graphs import build_graph
from IPython.display import Image, display

# 1. 그래프 생성
# 모드 선택: 'baseline', 'cot_previous', 'cot_upgrade' (IRP+SVI 모드)
mode = "cot_upgrade"
app_graph = build_graph(mode)

# 2. 그래프 시각화 (Mermaid 기반 PNG)
try:
    print(f"[{mode}] 모드의 랭그래프 시각화:")
    display(Image(app_graph.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print("그래프 이미지를 생성하는 중 에러가 발생했습니다. (라이브러리 종속성 문제일 수 있습니다.)\n", e)